# 22.2 一致但沒有資訊：collapse

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/22-collapse/)

先執行下一格的環境格，再閱讀、執行最後的完整實驗。各節互相獨立，不需要先跑其他節。

最後一格存著這段程式在 CPU 上實際執行的輸出；執行環境與其他實驗紀錄見網站的〈驗證範圍〉。範例不用預訓練權重，也不需要下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.6.0'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 各節實驗都在 CPU 上執行，執行紀錄用的是 PyTorch 2.9.1。已經是 2.9.1 就沿用（CPU 或 CUDA 版都可以）；
# 其他版本會改裝成 2.9.1 的 CPU 版，之後這個工作階段就用不到 GPU，但各節實驗本來就不需要。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
"""手工固定輸出的反例：跨 view CE 很低，仍可能完全無法區分圖片。"""
import json
from pathlib import Path

import torch

from miniyolo.self_distillation import DINOLoss, feature_diagnostics


def main():
    torch.set_num_threads(2)
    count, output_dim = 8, 16
    constant_features = torch.ones(count, 32)
    loss = DINOLoss(output_dim, student_temperature=1., teacher_temperature=1.)
    uniform = torch.zeros(count, output_dim)
    peaked = torch.full((count, output_dim), -10.)
    peaked[:, 0] = 10.
    results = {}
    for name, logits in (("constant_uniform", uniform), ("constant_peaked", peaked)):
        value = loss([logits, logits], [logits, logits])
        probabilities = loss.teacher_probabilities(logits)
        results[name] = {"cross_view_ce": float(value),
                         **feature_diagnostics(constant_features, probabilities)}
    # 相同 features 的 deterministic 分類器只會給所有圖同一答案；平衡兩類只對4/8。
    balanced_labels = torch.arange(count) % 2
    constant_predictions = torch.zeros(count, dtype=torch.long)
    results["constant_feature_balanced_color_accuracy"] = float((constant_predictions == balanced_labels).float().mean())
    results["constant_feature_balanced_color_correct"] = int((constant_predictions == balanced_labels).sum())
    results["constant_feature_balanced_color_count"] = count
    results["construction"] = "hand-written outputs, no learned model and no optimizer"
    results["interpretation"] = "low cross-view CE alone does not establish useful features; entropy/std/cosine describe different failures"
    results["centering_limit"] = "not an ablation: this does not show that disabling centering immediately collapses every training run"
    assert results["constant_peaked"]["cross_view_ce"] < 1e-5
    assert results["constant_peaked"]["feature_std"] == 0
    output = Path("artifacts/runs/dino")
    output.mkdir(parents=True, exist_ok=True)
    (output / "22-collapse.json").write_text(json.dumps(results, ensure_ascii=False, indent=2) + "\n")
    print(json.dumps(results, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


{
  "constant_uniform": {
    "cross_view_ce": 2.7725887298583984,
    "feature_std": 0.0,
    "normalized_feature_std": 0.0,
    "mean_pair_cosine": 0.9999999403953552,
    "mean_output_entropy": 2.7725887298583984,
    "marginal_output_entropy": 2.7725887298583984
  },
  "constant_peaked": {
    "cross_view_ce": 6.183461209730012e-07,
    "feature_std": 0.0,
    "normalized_feature_std": 0.0,
    "mean_pair_cosine": 0.9999999403953552,
    "mean_output_entropy": 6.183461209730012e-07,
    "marginal_output_entropy": 6.183461209730012e-07
  },
  "constant_feature_balanced_color_accuracy": 0.5,
  "constant_feature_balanced_color_correct": 4,
  "constant_feature_balanced_color_count": 8,
  "construction": "hand-written outputs, no learned model and no optimizer",
  "interpretation": "low cross-view CE alone does not establish useful features; entropy/std/cosine describe different failures",
  "centering_limit": "not an ablation: this does not show that disabling centering immediately col